# Training Model â€” audit delapan eksperimen

Notebook ini mendokumentasikan resep dan **artefak training yang sudah ada** untuk YOLO26m-seg, Swin-L + Mask2Former, EVA-02-L + Cascade Mask R-CNN, dan ECSeg-L; masing-masing pretrained dan scratch.

Sel di bawah membaca berkas log/CSV lokal untuk membuat ringkasan dan grafik. Sel tidak memulai training. Swin-L dan EVA sebelumnya dijalankan di Modal; YOLO26m dan ECSeg-L dijalankan lokal. Path dibuat relatif terhadap root proyek, jadi notebook bisa dibuka dari folder mana pun di dalam proyek.

> Angka validasi mengikuti `reports/MODEL_DIRECTORY.md`. Scratch Swin-L dan EVA berhenti manual, sedangkan ECSeg-L scratch selesai pada anggaran epoch-nya. Grafik hanya menggambarkan epoch yang tercatat, bukan klaim bahwa seluruh run selesai.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Cari root proyek dari direktori kerja notebook atau parent-nya.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "reports" / "MODEL_DIRECTORY.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Tidak menemukan reports/MODEL_DIRECTORY.md. Buka notebook dari dalam folder proyek.")

RUNS = {
    "YOLO26m pretrained": {"csv": "local_runs/k35_m_pf0/results.csv", "log": "local_runs/k35_m_pf0/train.log", "kind": "yolo"},
    "YOLO26m scratch": {"csv": "local_runs/k35_m_sf0_c3/results.csv", "log": "local_runs/k35_m_sf0_c3/train.log", "kind": "yolo"},
    "Swin-L pretrained": {"csv": "runs_local/m2f_v2/swinl35_f0_e150_b2_640_m20_r2/results.csv", "log": "runs_local/m2f_v2/logs/swinl35_f0_e150_b2_640_m20_r2_20261122-133258.txt", "kind": "swin"},
    "Swin-L scratch": {"csv": "runs_local/m2f_v2/swinl35_scratch_f0_e150_b2_640_m20_a2/results.csv", "log": "runs_local/m2f_v2/logs/swinl35_scratch_f0_e150_b2_640_m20_a2_20261122-233243.txt", "kind": "swin"},
    "EVA-02-L pretrained": {"csv": "artifacts/eva_f0_rpnfix/results.csv", "log": "artifacts/eva_f0_rpnfix/train.log", "kind": "eva"},
    "EVA-02-L scratch": {"csv": "artifacts/eva_f0_scratch20/results.csv", "log": "artifacts/eva_f0_scratch20/train.log", "kind": "eva"},
    "ECSeg-L pretrained": {"csv": None, "log": "reports/ecseg_l_k35_ws/logs/log.txt", "kind": "ecseg"},
    "ECSeg-L scratch": {"csv": None, "log": "reports/ecseg_l_k35_scratch/logs/log.txt", "kind": "ecseg"},
}

print(f"Root proyek: {ROOT}")
print(f"Jumlah run yang akan diringkas: {len(RUNS)}")

## Resep training dan cara menjalankan ulang

Perintah di bawah hanya disimpan sebagai referensi. Jangan jalankan ulang tanpa mengganti nama output dan memastikan GPU/Modal serta biaya tersedia. Checkpoint/log yang tercatat berasal dari eksperimen sebelumnya; konfigurasi finalnya ada di file run masing-masing.

YOLO26m memakai `scripts/train_k35.py`; ECSeg-L memakai config EdgeCrafter yang tersimpan. Swin-L dan EVA memakai entry point Modal di `modal/`. Baris perintah Modal hanya dokumentasi; notebook ini tidak mengirim job.

In [ ]:
# Referensi resep training (string saja; tidak dieksekusi oleh notebook).
TRAINING_RECIPES = {
    "YOLO26m pretrained": "python scripts/train_k35.py --model m --arm pretrained --fold 0 --epochs 24 --batch 4 --tag k35_m_pf0",
    "YOLO26m scratch": "python scripts/train_k35.py --model m --arm scratch --fold 0 --epochs 48 --batch 4 --resume-from local_runs/k35_m_sf0_c2/weights/last.pt --tag k35_m_sf0_c3",
    "Swin-L pretrained (Modal)": "modal run modal/m2f_train_v2.py --mode train --arm pretrained --epochs 150 --batch 2 --seed 0",
    "Swin-L scratch (Modal)": "modal run modal/m2f_train_v2.py --mode train --arm scratch --epochs 150 --batch 2 --seed 0",
    "EVA-02-L pretrained (Modal)": "modal run modal/eva_train.py --fold 0 --epochs 40 --batch 2 --pretrained",
    "EVA-02-L scratch (Modal)": "modal run modal/eva_train.py --fold 0 --epochs 120 --batch 2 --no-pretrained",
    "ECSeg-L pretrained": "cd third_party/EdgeCrafter/ecdetseg && python train.py -c configs/ecseg/ecseg_l_k35_ws.yml -t ecvits/ecseg_l.pth --seed 0",
    "ECSeg-L scratch": "cd third_party/EdgeCrafter/ecdetseg && python train.py -c configs/ecseg/ecseg_l_k35_scratch.yml --seed 0",
}

# Waspadai: perintah di atas adalah template dokumentasi, bukan jaminan
# bahwa semua argumen cocok untuk mereproduksi kondisi run lama secara bit-per-bit.
# Gunakan args.yaml/config, metrics.json, dan log run sebagai sumber kebenaran.
for name, command in TRAINING_RECIPES.items():
    print(f"{name}:\n  {command}\n")

## Fungsi baca log, ringkasan, dan kurva

Grafik menampilkan metrik yang benar-benar dicatat setiap training: YOLO menggunakan kolom mask mAP50-95 Ultralytics; Swin-L dan EVA menggunakan AP mask COCO; ECSeg-L mengambil AP/AP50 dari array `test_coco_eval_mask` pada log JSON. Skala metrik tidak sepenuhnya seragam untuk YOLO, sesuai catatan protokol di `MODEL_DIRECTORY.md`.

In [ ]:
def _load_run(name, spec):
    """Muat riwayat CSV atau log JSON ECSeg; tidak memulai training."""
    if spec["kind"] == "ecseg":
        log_path = ROOT / spec["log"]
        if not log_path.exists():
            raise FileNotFoundError(log_path)
        rows = []
        for line in log_path.read_text(encoding="utf-8").splitlines():
            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue
            ap = obj.get("test_coco_eval_mask")
            rows.append({"epoch": int(obj.get("epoch", len(rows))),
                         "train_loss": obj.get("train_loss"),
                         "val_AP": ap[0] if isinstance(ap, list) and len(ap) else None,
                         "val_AP50": ap[1] if isinstance(ap, list) and len(ap) > 1 else None})
        return pd.DataFrame(rows), log_path
    csv_path = ROOT / spec["csv"]
    log_path = ROOT / spec["log"]
    if not csv_path.exists():
        raise FileNotFoundError(csv_path)
    return pd.read_csv(csv_path), log_path


def _metric_columns(name, df, kind):
    cols = list(df.columns)
    if kind == "yolo":
        ap = next((c for c in cols if c.strip() == "metrics/mAP50-95(M)"), None)
        losses = [c for c in cols if c.strip().startswith("train/") and c.strip().endswith("_loss")]
        return ap, losses[:3]
    ap = next((c for c in ("val_AP", "val_best_ap") if c in cols), None)
    if kind == "ecseg":
        return "val_AP", ["train_loss"]
    loss = next((c for c in ("loss_mean", "loss", "train_loss") if c in cols), None)
    return ap, [loss] if loss else []


def show_run(name, spec):
    df, log_path = _load_run(name, spec)
    if df.empty:
        print(f"{name}: log/CSV tidak memiliki baris metrik.")
        return
    epoch_col = next((c for c in ("epoch", "Epoch") if c in df.columns), None)
    if epoch_col is None:
        df = df.copy()
        df["epoch"] = range(1, len(df) + 1)
        epoch_col = "epoch"
    ap_col, loss_cols = _metric_columns(name, df, spec["kind"])
    print(f"### {name} â€” {len(df)} baris tercatat")
    print(f"Log: {log_path.relative_to(ROOT)} ({'ada' if log_path.exists() else 'tidak ditemukan'})")
    view_cols = [c for c in [epoch_col, ap_col, *loss_cols] if c and c in df.columns]
    if view_cols:
        display(df[view_cols].tail(8).round(5))
    if log_path.exists():
        lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
        # ECSeg menulis satu JSON panjang per epoch; ringkas log itu ke metrik inti.
        if spec["kind"] == "ecseg":
            print("Cuplikan log (metrik inti tiap epoch terakhir):")
            for line in lines[-3:]:
                try:
                    obj = json.loads(line)
                    ap = obj.get("test_coco_eval_mask", [])
                    print({"epoch": obj.get("epoch"), "train_loss": round(obj.get("train_loss", 0), 4),
                           "val_AP": round(ap[0], 6) if ap else None,
                           "val_AP50": round(ap[1], 6) if len(ap) > 1 else None})
                except (json.JSONDecodeError, TypeError):
                    print(line[:250])
        else:
            print("Cuplikan akhir log:")
            print("\n".join(lines[-5:]))
    else:
        print("Log teks tidak tersedia; kurva dibaca dari CSV.")

    fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
    x = pd.to_numeric(df[epoch_col], errors="coerce")
    if ap_col and ap_col in df.columns:
        y = pd.to_numeric(df[ap_col], errors="coerce")
        valid = y.notna()
        axes[0].plot(x[valid], y[valid], marker="o", ms=3)
        axes[0].set_title(f"Validasi: {ap_col}")
        axes[0].set_ylabel("AP / mAP")
    else:
        axes[0].text(.5, .5, "Tidak ada AP validasi di CSV", ha="center", va="center")
        axes[0].set_title("AP validasi")
    axes[0].set_xlabel("Epoch")
    for col in loss_cols:
        if col in df.columns:
            y = pd.to_numeric(df[col], errors="coerce")
            valid = y.notna()
            if valid.any():
                axes[1].plot(x[valid], y[valid], label=col, linewidth=1.2)
    axes[1].set_title("Loss training")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    if axes[1].lines:
        axes[1].legend(fontsize=7)
    fig.suptitle(name)
    fig.tight_layout()
    plt.show()

## YOLO26m â€” pretrained dan scratch

Training dilakukan lokal dengan Ultralytics pada fold 0, 35 kelas, resolusi 640. Run pretrained tercatat 24 epoch; scratch merupakan tahap terakhir warm restart (`c3`) sepanjang 48 epoch pada run ini. Grafik membaca `results.csv`; log ditampilkan dari `train.log`.

In [ ]:
for _name in ("YOLO26m pretrained", "YOLO26m scratch"):
    show_run(_name, RUNS[_name])

## Swin-L + Mask2Former â€” pretrained dan scratch

Keduanya dijalankan di Modal pada L40S. Run pretrained berhenti lebih awal pada epoch 75/150; run scratch dihentikan manual pada epoch 68/150. `results.csv` dan log Modal adalah sumber grafik/cuplikan log.

In [ ]:
for _name in ("Swin-L pretrained", "Swin-L scratch"):
    show_run(_name, RUNS[_name])

## EVA-02-L + Cascade Mask R-CNN â€” pretrained dan scratch

Run pretrained dijalankan 40 epoch di Modal. Run scratch dihentikan manual di epoch 34 dari target 120; CSV mencatat pengukuran AP berkala, bukan setiap epoch. Jadi titik yang jarang pada grafik adalah sesuai frekuensi evaluasi sebenarnya.

In [ ]:
for _name in ("EVA-02-L pretrained", "EVA-02-L scratch"):
    show_run(_name, RUNS[_name])

## ECSeg-L â€” pretrained dan scratch

Kedua run lokal memakai EdgeCrafter dan 35 kelas. Pretrained warm-start dari bobot COCO; scratch memakai inisialisasi acak. Log ECSeg berbentuk JSON per epoch, sehingga helper mengekstrak `train_loss`, `test_coco_eval_mask[0]` (AP), dan `[1]` (AP50).

In [ ]:
for _name in ("ECSeg-L pretrained", "ECSeg-L scratch"):
    show_run(_name, RUNS[_name])

## Ringkasan hasil terdokumentasi

Nilai berikut disalin dari `reports/MODEL_DIRECTORY.md` dan bukan dihasilkan oleh sel training. Run scratch Swin-L dan EVA tidak selesai sesuai target; label statusnya perlu dipertahankan saat membandingkan kurva.

In [ ]:
summary = pd.DataFrame([
    ("YOLO26m", "pretrained", 0.3347, "mask mAP50-95; selesai, 24 epoch", "lokal"),
    ("YOLO26m", "scratch", 0.1016, "mask mAP50-95; selesai, 48 epoch pada run c3", "lokal"),
    ("Swin-L", "pretrained", 0.3265, "val mask AP; early-stop ep75/150", "Modal L40S"),
    ("Swin-L", "scratch", 0.000747, "best val mask AP; dihentikan manual ep68/150", "Modal L40S"),
    ("EVA-02-L", "pretrained", 0.322801, "val mask AP; complete 40/40", "Modal L40S"),
    ("EVA-02-L", "scratch", 0.016784, "best val mask AP; dihentikan manual ep34/120", "Modal L40S"),
    ("ECSeg-L", "pretrained", 0.4450, "segm AP; complete", "lokal"),
    ("ECSeg-L", "scratch", 0.004737, "best segm AP; complete 30/30", "lokal"),
], columns=["Model", "Arm", "AP", "Status / metrik", "Lokasi training"])
display(summary)